# nb128 — FULL Papyrus megafetch (not the ++ curated slice)

nb125 used `plusplus=True` (curated 19,948 records). The full Papyrus has ~60M activity records across thousands of targets. Filter the full set to PXR-related UniProts — expect 200k–1M records.

This is a much bigger pretraining corpus for multi-target Chemprop (nb129).

In [ ]:
import subprocess, sys, os, time
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'papyrus-scripts', 'rdkit', 'pystow'], check=False)
print('install done')

In [ ]:
from papyrus_scripts import download_papyrus
print('Downloading FULL Papyrus 05.7 (not ++)...')
download_papyrus(version='05.7', only_pp=False, structures=False, descriptors=None)
print('done')

In [ ]:
# Same target list as nb125
TARGETS = [
    'O75469', 'Q14994', 'P11473', 'Q96RI1', 'Q13133', 'P55055',
    'Q07869', 'P37231', 'Q03181',
    'P10276', 'P10826', 'P13631', 'P19793', 'P28702', 'P48443',
    'P10275', 'P03372', 'Q92731', 'P04150', 'P08235',
    'P10827', 'P10828', 'P41235', 'P11474', 'O95718', 'P62508',
    'P08684', 'P11712', 'P33261', 'P05177', 'P10635', 'P05181', 'P20815', 'P20813',
    'P08183', 'Q9UNQ0', 'Q92887', 'Q9Y6L6', 'Q9NPD5',
    'P35869', 'P02768',
]
print(f'{len(TARGETS)} targets')

In [ ]:
from papyrus_scripts.reader import read_papyrus
from papyrus_scripts.preprocess import keep_accession
import pandas as pd, time

t0 = time.time()
chunks = []
scanned = 0
for i, chunk in enumerate(read_papyrus(version='05.7', plusplus=False, is3d=False, chunksize=200_000)):
    sub = keep_accession(chunk, TARGETS)
    scanned += len(chunk)
    if len(sub) > 0:
        chunks.append(sub)
    if i % 10 == 0:
        kept = sum(len(c) for c in chunks)
        print(f'  chunk {i}: scanned {scanned:,}  kept {kept:,}  ({time.time()-t0:.0f}s)')

papy = pd.concat(chunks, ignore_index=True)
print(f'\nFull Papyrus filtered: {len(papy):,} rows x {papy.shape[1]} cols  in {time.time()-t0:.0f}s')
print(f'Targets covered: {papy["accession"].nunique()}')

In [ ]:
from pathlib import Path
out_dir = Path('/kaggle/working/papyrus_pxr_FULL')
out_dir.mkdir(parents=True, exist_ok=True)
# Convert object columns to str to avoid pyarrow type conflicts (e.g. all_years has mixed list/int)
for col in papy.columns:
    if papy[col].dtype == 'object':
        papy[col] = papy[col].astype(str)
papy.to_parquet(out_dir / 'papyrus_full_filtered.parquet', index=False)

# Quality filter: keep only records with high pchembl confidence (Quality not LOW)
# Then aggregate per (compound, target) for multi-target supervised use
value_col = 'pchembl_value_Mean'
smi_col = 'SMILES' if 'SMILES' in papy.columns else 'SMILES_Stripped'
print(f'Quality distribution:')
print(papy['Quality'].value_counts() if 'Quality' in papy.columns else 'N/A')

wide = papy.pivot_table(index=smi_col, columns='accession', values=value_col, aggfunc='median')
wide.reset_index().to_parquet(out_dir / 'papyrus_full_wide.parquet', index=False)
print(f'Wide pivot: {wide.shape}')

summary = papy.groupby('accession').agg(
    n_records=(value_col, 'count'),
    mean_pchembl=(value_col, 'mean'),
    std_pchembl=(value_col, 'std'),
).sort_values('n_records', ascending=False)
summary.to_csv(out_dir / 'target_summary_full.csv')
print(summary.head(20))